# 02 - Train the context encoder (homograph disambiguation)

This is the cheap, high-value stage: about 20 minutes on a 4090. It is also
where homograph accuracy actually comes from, so iterate here before spending
money on the acoustic model.

The model is about 6M parameters. For every word it predicts which of that
word's discovered readings applies in this context.

In [1]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp0_small.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

repo   : g:\Adaptive-TTS
config : configs/exp0_small.yaml -> exp0_small_egyptian
dataset: OmarAhmedSobhy/tts-egyption-dataset
probe  : الدول دول علم مصر


## Start TensorBoard

Watch `eval/code_acc`. That is held-out homograph accuracy, the number that
matters. Above roughly 0.90 means the approach is working.

In [2]:
%load_ext tensorboard
%tensorboard --logdir $cfg.paths.tb_dir --port 6006 --bind_all

## Train

In [3]:
!python scripts/train_context.py --config $CONFIG

^C


## Evaluate what it learned

The two `علم` sentences must get *different* codes. If they do not,
disambiguation is not working and the acoustic model will inherit the failure.

In [ ]:
import json
from adaptts.infer.pipeline import AdapTTS

tts = AdapTTS.from_checkpoints(CONFIG, device="cpu", load_codec=False)
probes = json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]

for p in probes:
    plan = tts.analyze(p["text"])
    hard = [f"{w.word}=code{w.code}({w.confidence:.2f})" for w in plan.hard_words]
    print()
    print(p["tag"])
    print("  text      :", p["text"])
    print("  expected  :", p["expected"])
    print(f"  difficulty: {plan.sentence_difficulty:.3f} -> depth {plan.depth}")
    print("  decisions :", ", ".join(hard) if hard else "no ambiguous words")

In [ ]:
# The critical comparison: one word, two contexts, two readings.
a = tts.analyze("انا شوفت علم مصر بيرفرف")             # flag
b = tts.analyze("علم الفيزيا من اهم العلوم البشرية")   # science
ca = [w.code for w in a.hard_words if w.word == "علم"]
cb = [w.code for w in b.hard_words if w.word == "علم"]
print("flag context    -> code", ca)
print("science context -> code", cb)
print()
print("DISAMBIGUATION WORKS" if ca and cb and ca != cb
      else "NOT disambiguating: investigate before training the acoustic model")

In [ ]:
# The multi-homograph stress sentence from the brief.
plan = tts.analyze(
    "انا كنت مصر على ان مصر عندها امكانيات و موارد تخليها تتفوق على دول من اللي شايفين نفسهم دول"
)
print(plan)

## Inference speed on CPU

The context encoder must be negligible next to the acoustic model.

In [ ]:
import time

txt = "انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول"
for _ in range(3):
    tts.analyze(txt)                      # warm up
t0 = time.perf_counter()
for _ in range(50):
    tts.analyze(txt)
print(f"context encoder: {(time.perf_counter() - t0) / 50 * 1000:.2f} ms per sentence on CPU")

If accuracy looks good, continue to **03_train_acoustic.ipynb**.